### Algorithms matter

Let's compute $a ^ 2 - b ^ 2$ numerically using 2 different algorithms.

#### First algorithm

1. Compute $a^2$
2. Compute $b^2$
3. Compute $a^2$ - $b^2$

#### Second algorithm

1. Compute $a + b$
2. Compute $a - b$
3. Compute $(a + b) \cdot (a - b)$

We're interested in the total relative error (inherited error + algorithmic error). Remember that:

$$
    z_{machine} = z \, (1 + \varepsilon_z) \qquad \left| \varepsilon_z \right| \leq u \approx 1.1 \times 10^{- 16}
$$

In [1]:
import sympy as smp

In [2]:
a, b = smp.symbols('a, b', real = True, positive = True) # a, b
eps_a, eps_b, eps_z1, eps_z2, eps_z3 = smp.symbols('\\varepsilon_{a}, \\varepsilon_{b}, \\varepsilon_{z1}, \\varepsilon_{z2}, \\varepsilon_{z3}', real = True, nonzero = True) # Relative errors
t = smp.symbols('t', real = True) # Dummy variable

In [3]:
def fl(z, eps_z): # Floating operator
    return z * (1 + eps_z)

#### First algorithm

In [4]:
F = a ** 2 - b ** 2 # Exact result (symbolically)

z1 = fl(fl(a, eps_a) * fl(a, eps_a), eps_z1) # Computing a^2
z2 = fl(fl(b, eps_b) * fl(b, eps_b), eps_z2) # Computing b^2
res = fl(z1 - z2, eps_z3) # Final result
res

(\varepsilon_{z3} + 1)*(a**2*(\varepsilon_{a} + 1)**2*(\varepsilon_{z1} + 1) - b**2*(\varepsilon_{b} + 1)**2*(\varepsilon_{z2} + 1))

In [5]:
res = res.expand().subs({e: e * t for e in [eps_a, eps_b, eps_z1, eps_z2, eps_z3]}).series(t, 0, 2).removeO().subs(t, 1) # Wiping out all the second order terms
res

2*\varepsilon_{a}*a**2 - 2*\varepsilon_{b}*b**2 + \varepsilon_{z1}*a**2 - \varepsilon_{z2}*b**2 + \varepsilon_{z3}*a**2 - \varepsilon_{z3}*b**2 + a**2 - b**2

In [6]:
err = ((res - F) / F).expand() # Total relative error committed (inherited + algorithmic)
err

2*\varepsilon_{a}*a**2/(a**2 - b**2) - 2*\varepsilon_{b}*b**2/(a**2 - b**2) + \varepsilon_{z1}*a**2/(a**2 - b**2) - \varepsilon_{z2}*b**2/(a**2 - b**2) + \varepsilon_{z3}*a**2/(a**2 - b**2) - \varepsilon_{z3}*b**2/(a**2 - b**2)

The upper bound of this quantity is:

$$
    \left| \varepsilon_{F} \right| \lesssim \left( 3 \, \frac{a^2 + b^2}{\left| a^2 -  b^2 \right|} + 1 \right) \, u
$$

#### Second algorithm

In [7]:
z1 = fl(fl(a, eps_a) + fl(b, eps_b), eps_z1) # Computing a + b
z2 = fl(fl(a, eps_a) - fl(b, eps_b), eps_z2) # Computing a - b
res = fl(z1 * z2, eps_z3) # Final result
res

(\varepsilon_{z1} + 1)*(\varepsilon_{z2} + 1)*(\varepsilon_{z3} + 1)*(a*(\varepsilon_{a} + 1) - b*(\varepsilon_{b} + 1))*(a*(\varepsilon_{a} + 1) + b*(\varepsilon_{b} + 1))

In [8]:
res = res.expand().subs({e: e * t for e in [eps_a, eps_b, eps_z1, eps_z2, eps_z3]}).series(t, 0, 2).removeO().subs(t, 1) # Keep only first order terms
res

2*\varepsilon_{a}*a**2 - 2*\varepsilon_{b}*b**2 + \varepsilon_{z1}*a**2 - \varepsilon_{z1}*b**2 + \varepsilon_{z2}*a**2 - \varepsilon_{z2}*b**2 + \varepsilon_{z3}*a**2 - \varepsilon_{z3}*b**2 + a**2 - b**2

In [9]:
err = ((res - F) / F).expand() # Total relative error committed (inherited + algorithmic)
err

2*\varepsilon_{a}*a**2/(a**2 - b**2) - 2*\varepsilon_{b}*b**2/(a**2 - b**2) + \varepsilon_{z1}*a**2/(a**2 - b**2) - \varepsilon_{z1}*b**2/(a**2 - b**2) + \varepsilon_{z2}*a**2/(a**2 - b**2) - \varepsilon_{z2}*b**2/(a**2 - b**2) + \varepsilon_{z3}*a**2/(a**2 - b**2) - \varepsilon_{z3}*b**2/(a**2 - b**2)

This time the upper bound is different, and is equal to:

$$
    \left| \varepsilon_{F} \right| \lesssim \left( 2 \, \frac{a^2 + b^2}{\left| a^2 -  b^2 \right|} + 3 \right) \, u
$$

When $a \approx b$, the second algorithm is better than the previous one

In [10]:
av = 1.23456789
bv = av - 1e-15 # a - b = 10 ^ {- 15}

u = 1.1e-16 # roundoff error

In [11]:
# First algorithm
z1 = av * av
z2 = bv * bv
res = z1 - z2
res, u * (1.0 + (3.0 * ((av ** 2 + bv ** 2)/ abs(res)))) * 100.0 # Result and relative error [%]

(2.886579864025407e-15, 34.84899933133184)

In [12]:
# Second algorithm
z1 = av + bv
z2 = av - bv
res = z1 * z2
res, u * (3.0 + (2.0 * ((av ** 2 + bv ** 2)/ abs(res)))) * 100.0 # Result and relative error [%]

(2.7412913938817934e-15, 24.463997753217367)